In [1]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

Cloning into 'capstone_project_GroupA'...
remote: Enumerating objects: 1209, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 1209 (delta 191), reused 184 (delta 88), pack-reused 884 (from 3)
Receiving objects: 100% (1209/1209), 331.48 MiB | 39.99 MiB/s, done.
Resolving deltas: 100% (597/597), done.
Updating files: 100% (99/99), done.


In [2]:
%cd capstone_project_GroupA
!git checkout colab
%cd src

/content/capstone_project_GroupA
Branch 'colab' set up to track remote branch 'colab' from 'origin'.
Switched to a new branch 'colab'
/content/capstone_project_GroupA/src


In [ ]:
from datetime import datetime

from ModelFiles.GroupAModels import GradientBoostingModel
from ModelFiles.ModelConfigs import GradientBoostingConfig, HORIZONS, SEEDS
from ModelFiles.ModelPlots import *
from ModelFiles.LSTM.LSTMUtils import add_lag_features # shared utility function with LSTM

CONTEXT_LENGTH = None # GB we are fitting entire results set. Need to find literature to support this though.
USE_LOG_TARGET = True
EVAL_STEP_SIZE = 48
N_ESTIMATORS = [100,200,300,600,1000,1500]
SEEDS = [27182, 14142, 17320, 22360, 57721]
ONE_SEED = [31415]
N_DEPTH = [1, 3, 5, 7]
DEBUG = False

for n_estimator in N_ESTIMATORS:
    for max_depth in N_DEPTH:
        for horizon in HORIZONS:
            for seed in ONE_SEED:#SEEDS:
                if seed == ONE_SEED[-1]:#SEED[-1]:
                    save_prediction_results = True
                else:
                    save_prediction_results = False

                gb_config = GradientBoostingConfig(
                    task_id=f"gb_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                    forecast_horizon=horizon,
                    lookback_window=CONTEXT_LENGTH, # For Gradient Boosting, this is the number of most recent time steps to use for training.
                    target_col='LOG_TOTALDEMAND' if USE_LOG_TARGET else 'TOTALDEMAND',
                    target_lags=[1,2,3,48,96],
                    target_mas=[6, 48],
                    used_log_target=USE_LOG_TARGET,
                    feature_cols=['TEMPERATURE','TEMP_SQUARED', 'HOUR', 'DAYOFWEEK', 'IS_WEEKEND'],
                    n_estimators=n_estimator,
                    learning_rate=0.1,
                    max_depth=max_depth,
                    verbose=1,
                    scale=True,
                    seed=seed,
                    eval_step_size=EVAL_STEP_SIZE, # This is the step size to use when evaluating against validation or test set.
                    debug=DEBUG,
                    save_training_log=True,
                    save_test_results=save_prediction_results,
                )
                print(f"\nRunning Gradient Boosting with config: {gb_config}\n")
                gb_model = GradientBoostingModel(gb_config, add_lag_features)
                gb_model.train_model()
                all_actuals, all_predictions, rmse, mae = gb_model.evaluate_model(None, test_mode=1)
                print("=" * 200)
                print("\n")

Found NSW data path: /content/capstone_project_GroupA/data/NSW


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Set random seed to 31415
Use GPU: cuda:0
train 107300
	iters: 100, epoch: 1 | loss: 0.2150693
	speed: 0.0510s/iter; left time: 17078.9290s
	iters: 200, epoch: 1 | loss: 0.2492466
	speed: 0.0200s/iter; left time: 6691.5602s
